# 📊 Product Risk Analysis

This project analyzes product return behavior and financial impact using transactional retail data.

The goal is to identify high-risk products that generate high return losses.

## Import Libraries

In [4]:
import pandas as pd
import numpy as np
import re

## Data Loading

The dataset is loaded and prepared for analysis.

In [5]:
df = pd.read_csv(r"data\online_retail_II.csv",
                 sep=";",
                 decimal=",",
                 encoding="ISO-8859-1")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,1.12.2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,1.12.2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,1.12.2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,1.12.2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,1.12.2010 08:26,3.39,17850.0,United Kingdom


## Data Information

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  object 
 1   StockCode    541910 non-null  object 
 2   Description  540456 non-null  object 
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  object 
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [4]:
df.describe()

,Quantity,Price,Customer ID
count,541910.000000,541910.000000,406830.000000
mean,9.552234,4.611138,15287.684160
std,218.080957,96.759765,1713.603074
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [5]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

In [6]:
df.dropna(inplace=True)

## Country Distribution

We analyze transaction distribution by country to identify the dominant market.

In [24]:
df["Country"].value_counts().head()

Country
United Kingdom    361878
Germany             9495
France              8492
EIRE                7485
Spain               2533
Name: count, dtype: int64

## Country Selection

The United Kingdom represents the majority of transactions.
To ensure consistency, the analysis focuses on UK data only.

In [8]:
df_uk = df[df["Country"] == "United Kingdom"].copy()

## Data Cleaning

We clean product descriptions and remove invalid or non-product records such as operational entries (e.g., postage, discounts).

This ensures that the analysis focuses only on real product transactions.

In [12]:
df_uk["Description_clean"]=(
    df_uk["Description"]
.astype(str)
.str.upper()
.str.strip())

invalid_values = ["", "?", "??", "???",
                  "MISSING", "UNKNOWN", "N/A", "NONE", "NULL"]
df_uk = df_uk[~df_uk["Description_clean"].isin(invalid_values)]

exclude_words = [
    "POSTAGE", "DISCOUNT", "BANK", "CHARGES", "COMMISSION",
    "MANUAL", "ADJUST", "FEE", "CRUK", "SAMPLES",
    "CHECK", "DAMAGED", "WRONG", "WRONGLY", "INCORRECTLY", "DOTCOM", "TEST"
]

pattern_exclude = re.compile(
    r"\b(?:" + "|".join(map(re.escape, exclude_words)) + r")\b"
)

df_uk = df_uk[~df_uk["Description_clean"].str.contains(
    pattern_exclude, na=False)]

pattern_valid = r"^[A-Z0-9 \-.,'/]+$"
df_uk = df_uk[df_uk["Description_clean"].str.match(pattern_valid, na=False)]

df_uk = df_uk[df_uk["Description_clean"].str.len() > 3]




## Feature Engineering

We calculate:
- Revenue
- Sales Revenue
- Return Revenue

These metrics help us understand both sales performance and return behavior.

In [13]:
df_uk["Revenue"] = df_uk["Price"] * df_uk["Quantity"]
df_uk["Sales_Revenue"] = np.where(df_uk["Quantity"] > 0, df_uk["Revenue"], 0)
df_uk["Return_Revenue"] = np.where(
    df_uk["Quantity"] < 0, abs(df_uk["Revenue"]), 0)



## Product-Level Aggregation

We aggregate data at the product level to calculate:
- Total sold quantity
- Total returned quantity
- Sales revenue
- Return revenue
- Return rate

In [14]:
product_stats = df_uk.groupby("Description_clean").agg(
    total_sold=("Quantity", lambda x: x[x > 0].sum()),
    total_returned=("Quantity", lambda x: abs(x[x < 0].sum())),
    sales_revenue=("Sales_Revenue", "sum"),
    return_revenue=("Return_Revenue", "sum")
).fillna(0).reset_index()

product_stats["return_rate"] = np.where(product_stats["total_sold"] == 0,
                                        np.nan,
                                        (product_stats["total_returned"] /
                                         product_stats["total_sold"])*100
                                        ).round(2)




## Impact Score

Impact score measures the financial impact of returns by combining return rate and return revenue.

This helps prioritize products that generate the highest loss.

In [ ]:
product_stats["impact_score"] = (
    product_stats["return_revenue"] *
    (product_stats["return_rate"] / 100)
).round(2)

## Anomaly Detection

We identify unusual patterns such as:
- Returns without sales
- Returns exceeding sales
- Fully reversed transactions

These are flagged as data issues

In [15]:
def classify_anomaly(row):
    if row["total_sold"] == row["total_returned"] and row["total_sold"] > 0:
        return "Operational Reversal"
    elif row["total_sold"] == 0 and row["total_returned"] > 0:
        return "Suspicious Return"
    elif row["total_returned"] > row["total_sold"]:
        return "Data Issues"
    else:
        return "Normal"


product_stats["anomaly_flag"] = product_stats.apply(classify_anomaly, axis=1)

print(product_stats["anomaly_flag"].value_counts())


anomaly_flag
Normal                  3820
Suspicious Return         16
Data Issues                7
Operational Reversal       6
Name: count, dtype: int64


## Top Risky Products

Products are ranked based on their financial impact (impact score).

This highlights items that generate the highest return losses.

In [21]:
top_risk_products = product_stats.sort_values(
    by="impact_score", ascending=False
).head(10)

top_risk_products

,Description_clean,total_sold,total_returned,sales_revenue,return_revenue,return_rate,anomaly_flag,impact_score
2299,"PAPER CRAFT , LITTLE BIRDIE",80995,80995,168469.60,168469.60,100.00,Operational Reversal,168469.60
1975,MEDIUM CERAMIC TOP STORAGE JAR,76919,74467,80291.44,77446.31,96.81,Normal,74975.77
1966,MANUAL,7137,3456,20198.94,75058.35,48.42,Normal,36343.25
2584,POSTAGE,38,59,8784.02,9537.99,155.26,Data Issues,14808.68
2277,PANTRY CHOPPING BOARD,1047,946,5274.21,4803.06,90.35,Normal,4339.56
3391,TEA TIME PARTY BUNTING,2118,1424,9653.10,3692.95,67.23,Normal,2482.77
1146,FAIRY CAKE FLANNEL ASSORTED COLOUR,9711,3150,16770.36,6591.42,32.44,Normal,2138.26
1186,FELTCRAFT DOLL MOLLY,4017,1446,10530.39,3509.70,36.00,Normal,1263.49
998,DOORMAT FAIRY CAKE,3332,672,20427.64,4538.40,20.17,Normal,915.40
1246,FLOWERS CHANDELIER T-LIGHT HOLDER,809,435,2474.69,1504.35,53.77,Normal,808.89


## Export for Power BI

The full dataset is exported to Power BI without filtering anomalies.

Risk level classification and filtering are handled within Power BI to allow dynamic analysis and interactivity.

In [ ]:
product_stats.to_csv(
    "product_analysis.csv",
    sep=";",
    decimal=",",
    index=False)


## Risk Classification (Python Version)

For comparison purposes, risk levels are also calculated in Python using only normal transactions.

In [17]:
clean_products = product_stats[product_stats["anomaly_flag"] == "Normal"].copy()


In [18]:
def classify_risk(rate):
    if rate >= 70:
        return "Critical"
    elif rate >= 40:
        return "High"
    elif rate >= 20:
        return "Medium"
    else:
        return "Low"


clean_products["risk_level"] = clean_products["return_rate"].apply(
    classify_risk)

print(clean_products["risk_level"].value_counts())


risk_level
Low         3729
Medium        53
High          33
Critical       5
Name: count, dtype: int64
